# Notebooke 2: Use MCTS with ML evaluators to generate chemical compounds

This notebook demonstrates how to manipulate MCTS search parameters to generate molecules wtih a more sophisticated chemical evaluator, MinervaChem.

Set a seed for reproducible MCTS results.

In [1]:
import random
seed = 42
random.seed(seed)

In [ ]:
import sys
sys.path.insert(1, '../../minervachem/mcts')
from tree import *
from tree_viz import *

## 1: Setting MCTS search parameters

As shown in Notebook 1, we can set MCTS's search parameters for any specific scenario. (See Notebook 1 for a full explanation of each parameter.)

In this example, we are using a MinervaChem ML model to optimized for atomization energy.

In [4]:
goal = -2000 # target atomization energy value
max_value1 = -1000 # max atomization energy

sa_target = 0 # target synthesizability score (sa score)
max_value2 = 5 # max sa score

# SMILES symbols that are choices for molecular generation
choices = ['C', 'O', '=', 'N', 'c', '1', 'F', '\n']

levels = 5 # size of molecule

num_sims = 10000 # number of simulations, default 10000; set smaller to test code
turn = levels + 4 # counter to track number of turns
beamsize = 10 # set the beamsize
alpha = 2 # boltzmann constant

## 2: Run MCTS with MinervaChem

Because MinervaChem is a more complex evaluator, this extends MCTS's run time from <30 seconds with RDKit, to >1 minute.

In [ ]:
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*') # these lines are to silence RDKit warnings from invalid molecules

current_nodes = [Node(BondEnergy(
    goal=goal, 
    sa_target=sa_target, 
    allchoices=choices,
    max_value1=max_value1,
    turn=turn))]
for l in range(levels):
    next_nodes=utcbeam(budget=num_sims,rootpop=current_nodes, beamsize=beamsize, alpha=alpha, scalar=0.4)
    for i in current_nodes:
        print(f"Level {l}")
        print(f"This is one of the current nodes: {i}")
        print(f"Num Children: {len(i.children)}")
        for j,c in enumerate(i.children):
            print(j,c)
    print("These are the best children:")
    for i in next_nodes:
        print(i)
    current_nodes = next_nodes
    print("--------------------------------")

We can display the grpahical representations of the generated molecules.

In [ ]:
allsmiles = []
allmols = []

for i in current_nodes:
    smiles = i.state.smiles
    allsmiles += [smiles]

    mol = Chem.MolFromSmiles(smiles)
    allmols.append(mol)
    mol.SetProp("name", str(Chem.rdMolDescriptors.CalcMolFormula(mol)))
    mol.SetProp("logP", str(f"{Descriptors.MolLogP(mol):.3f}"))
    mol.SetProp("SA score", str(f"{sascorer.calculateScore(mol):.3f}"))

img = Chem.Draw.MolsToGridImage(
    allmols, 
    legends=[f"{mol.GetProp('name')}" for mol in allmols],
    subImgSize=(350,350))

img

## 3: Plot tree searches

We can visualize the search paths with the same visualization module used in Notebook 1. Options for plotting values are ```'visits'```, ```'reward'```, ```'e_at'```, and ```'sa_score'```.

In [ ]:
for i, j in enumerate(current_nodes):
    tuples = make_tree_nodes(node=j, size=levels)
    node_info = make_node_info(node=j, size=levels)
    # display(node_info)
    mass_plotting(node_info=node_info, params=['e_at'], tuples=tuples, smiles=allsmiles[i])